# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and analyzing the FAIR^2 dataset using the `mlcroissant` library. All entities (record sets, fields, columns) are referenced by their `@id` following best practices.

### Dataset Source
The dataset Croissant schema is provided at:
```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

Explore this notebook step-by-step to:
1. Load the dataset metadata and records
2. Review available record sets, fields, and their `@id`s
3. Extract records into DataFrames for analysis
4. Perform EDA and basic data processing
5. Visualize and summarize the data

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {getattr(metadata, 'name', None)}\nDescription: {getattr(metadata, 'description', None)}")

## 2. Data Overview
Review the available record sets, fields, and their `@id` identifiers. Each record set encapsulates a logical table of data. We will enumerate record sets, list each one's field IDs, and show the first record of each for context.

In [ ]:
# List all record sets by @id, field IDs, and show an example record.
record_sets = metadata.record_sets
print(f"Number of record sets: {len(record_sets)}")
for rs in record_sets:
    print(f"---\nRecordSet name: {getattr(rs, 'name', None)}\n@id: {getattr(rs, '@id', None)}")
    field_ids = [getattr(field, '@id', None) for field in getattr(rs, 'fields', [])]
    print(f"Field @ids: {field_ids}")
    # Show the first record, if it exists
    try:
        records = list(dataset.records(record_set=getattr(rs, '@id', None)))
        if records:
            print("Example record:")
            # Only print small dicts
            if len(records[0]) < 10:
                print(records[0])
            else:
                print({k: records[0][k] for k in list(records[0].keys())[:5]})
        else:
            print("No records found.")
    except Exception as e:
        print(f"Could not load records: {e}")

## 3. Data Extraction
Load records from each record set into a pandas DataFrame. This allows flexible analysis by referencing the record set and field `@id`s. We collect one DataFrame for each record set.

In [ ]:
# Map DataFrames by record set @id
dataframes = dict()
record_set_ids = [getattr(rs, '@id') for rs in record_sets]
for rsid in record_set_ids:
    records = list(dataset.records(record_set=rsid))
    df = pd.DataFrame(records)
    dataframes[rsid] = df
    print(f"RecordSet {rsid} - columns: {df.columns.tolist()}")
    print(df.head(2))
    print("")

# For further analysis, pick the main record set (likely the one with the clinical CRC variables)
# As of 2024-06, this dataset typically contains one main data table record set. Replace the string below with the actual @id if needed.
main_record_set_id = record_set_ids[0]  # Use the first record set by default
df = dataframes[main_record_set_id]
print(f"Selected main record set @id: {main_record_set_id}")
df.head()

## 4. Exploratory Data Analysis (EDA)
We'll process the main record set DataFrame, demonstrate filtering a numeric field, normalizing, and grouping. All field and column accesses use their `@id` for clarity.

In [ ]:
# Find all numeric fields in the main record set (by Croissant dataType)
main_rs = [rs for rs in record_sets if getattr(rs, '@id') == main_record_set_id][0]
numeric_fields = [
    getattr(field, '@id') for field in getattr(main_rs, 'fields', [])
    if getattr(field, 'data_type', None) in ['schema:Integer', 'schema:Float', 'schema:Number']
    and getattr(field, '@id') in df.columns
]

print(f"Numeric fields for analysis (@id): {numeric_fields}")
# Select a numeric field; here we default to the first found
numeric_field_id = numeric_fields[0] if numeric_fields else None
assert numeric_field_id is not None, "No numeric fields found."

# Demonstrate filtering
threshold = df[numeric_field_id].median() if not df[numeric_field_id].isnull().all() else 0
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered {len(filtered_df)} records with {numeric_field_id} > {threshold} (median)")
print(filtered_df[[numeric_field_id]].head())

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by a categorical field (e.g., sex or cancer type)
categorical_fields = [
    getattr(field, '@id') for field in getattr(main_rs, 'fields', [])
    if getattr(field, 'data_type', None) == 'schema:Text'
    and getattr(field, '@id') in df.columns
]

group_field_id = categorical_fields[0] if categorical_fields else None
if group_field_id:
    grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean')
    print(f"Mean {numeric_field_id} grouped by {group_field_id}:")
    print(grouped.head())

## 5. Visualization
Let's plot the distribution of the chosen numeric field and compare it across a grouping field if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field_id].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Boxplot by group (if a categorical field was found)
if group_field_id:
    plt.figure(figsize=(8,4))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
- We loaded and parsed a clinical oncology dataset using `mlcroissant`, referencing all entities by their Croissant `@id`.
- We explored and described available record sets and their fields.
- We extracted tabular data, applied filtering and normalization, grouped by categories, and visualized a numeric field's distribution.
- These methods provide a reproducible workflow for FAIR data processing and EDA in biomedical research using Croissant-based datasets.